In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import time
start = time.time()

import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ADS_Project/data/raw/train.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)
df.head()

end = time.time()
print(f"Total time: {end - start:.2f} seconds")


Total time: 3.56 seconds


In [3]:
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)


In [4]:
LAGS = [1, 7, 14, 30]

for lag in LAGS:
    df[f"sales_lag_{lag}"] = (
        df.groupby(["store", "item"])["sales"].shift(lag)
    )


In [5]:
df["rolling_mean_7"] = (
    df.groupby(["store", "item"])["sales"]
      .shift(1)
      .rolling(7)
      .mean()
)

df["rolling_mean_14"] = (
    df.groupby(["store", "item"])["sales"]
      .shift(1)
      .rolling(14)
      .mean()
)


In [6]:
df.isna().sum().sort_values(ascending=False).head()

,0
sales_lag_30,15000
sales_lag_14,7000
rolling_mean_14,7000
sales_lag_7,3500
rolling_mean_7,3500


In [7]:
df_fe = df.dropna().reset_index(drop=True)


In [8]:
df_fe.to_csv("/content/drive/MyDrive/Colab Notebooks/ADS_Project/data/processed/feature_data.csv", index=False)


In [9]:
df_fe.head()


,date,store,item,sales,day,month,year,day_of_week,is_weekend,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_30,rolling_mean_7,rolling_mean_14
0,2013-01-31,1,1,13,31,1,2013,3,0,9.0,8.0,16.0,13.0,10.285714,10.857143
1,2013-02-01,1,1,11,1,2,2013,4,0,13.0,14.0,7.0,11.0,11.000000,10.642857
2,2013-02-02,1,1,21,2,2,2013,5,1,11.0,12.0,18.0,14.0,10.571429,10.928571
3,2013-02-03,1,1,15,3,2,2013,6,1,21.0,12.0,15.0,13.0,11.857143,11.142857
4,2013-02-04,1,1,14,4,2,2013,0,0,15.0,11.0,8.0,10.0,12.285714,11.142857
